# PLDM - Planning with Latent Dynamics Models on Google Colab

このノートブックは、論文 ["Learning from Reward-Free Offline Data: A Case for Planning with Latent Dynamics Models"](https://arxiv.org/abs/2502.14819) のコードをGoogle Colab上で実行するためのセットアップガイドです。

- [Paper](https://arxiv.org/abs/2502.14819)
- [Website](https://latent-planning.github.io/)
- [GitHub Repository](https://github.com/vladisai/PLDM)

## 1. GPU設定の確認

GPUが利用可能か確認します。このプロジェクトはGPUを推奨します。

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU not available. Training will be slow.")
    print("Colab: Runtime → Change runtime type → GPU を選択してください")

## 2. リポジトリのクローン

In [ ]:
# 既存のディレクトリを削除して新しくクローン
!rm -rf PLDM
!git clone https://github.com/vladisai/PLDM.git
%cd PLDM

## 3. 依存パッケージのインストール

必要なPythonパッケージをインストールします。数分かかる場合があります。

In [ ]:
# Colab環境に合わせてPyTorchとtorchvisionを先にインストール
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121

# requirements.txtから依存パッケージをインストール
!pip install -r requirements.txt

# パッケージを開発モードでインストール
!pip install -e .

## 4. インストールの確認

In [ ]:
# 主要なパッケージのインポート確認
import torch
import jax
import flax
import gym
import gymnasium
import wandb
import omegaconf

print("✓ All packages imported successfully!")
print(f"PyTorch: {torch.__version__}")
print(f"JAX: {jax.__version__}")
print(f"Flax: {flax.__version__}")

## 5. Weights & Biases (WandB) の設定（オプション）

実験のログを記録する場合は、WandBにログインします。スキップする場合は、トレーニング時に `--values wandb=False` を指定してください。

In [ ]:
# WandBへのログイン（オプション）
# APIキーを入力してください: https://wandb.ai/authorize
!wandb login

## 6. データセットのセットアップ

### オプション A: Wall環境（Two Rooms）のデータセット

In [ ]:
# Wall環境のデータセットをダウンロード
!bash pldm_envs/wall/presaved_datasets/download_all.sh

# 画像のレンダリング（必要な場合）
!bash pldm_envs/wall/presaved_datasets/render_all.sh

### オプション B: Diverse Maze環境のデータセット

In [ ]:
# 元の論文のデータセットを生成
!bash pldm_envs/diverse_maze/data_generation/generate_all_datasets_og.sh

# または新しいデータセット（40/20/10/5 maps）を生成
# !bash pldm_envs/diverse_maze/data_generation/generate_all_datasets_new.sh

## 7. トレーニングの実行

### クイックデバッグモード（短時間でテスト）

In [ ]:
# Wall環境でクイックデバッグ（WandBなし）
!python pldm/train.py \
  --config pldm/configs/wall/icml/seqlen90_3M.yaml \
  --values quick_debug=True wandb=False

### フルトレーニング

In [ ]:
# Wall環境でフルトレーニング（sequence length 90, 3M dataset）
!python pldm/train.py --config pldm/configs/wall/icml/seqlen90_3M.yaml

In [ ]:
# Diverse Maze環境でフルトレーニング（5 maps setting）
!python pldm/train.py --config pldm/configs/diverse_maze/icml/small_diverse_5maps.yaml

### カスタム設定でのトレーニング

In [ ]:
# コマンドラインから設定をオーバーライド
!python pldm/train.py \
  --config pldm/configs/wall/icml/seqlen90_3M.yaml \
  --values base_lr=0.01 data.offline_wall_config.batch_size=128

## 8. 評価の実行

学習済みチェックポイントを使って評価を実行します。

In [ ]:
# チェックポイントパスを指定して評価のみ実行
# !python pldm/train.py \
#   --config pldm/configs/wall/icml/seqlen90_3M.yaml \
#   --values eval_only=True load_checkpoint_path=/path/to/checkpoint.pt

## 9. 結果の可視化

トレーニング中の損失やメトリクスはWandBで確認できます。

また、保存されたチェックポイントやログは以下のディレクトリに保存されます：

In [ ]:
# チェックポイントの確認
!ls -lh checkpoints/ 2>/dev/null || echo "チェックポイントはまだ保存されていません"

# ログの確認
!ls -lh logs/ 2>/dev/null || echo "ログはまだ保存されていません"

## 10. Google Driveへの保存（オプション）

トレーニング結果をGoogle Driveに保存する場合：

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# チェックポイントをGoogle Driveにコピー
!mkdir -p /content/drive/MyDrive/PLDM_checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/PLDM_checkpoints/ 2>/dev/null || echo "コピーするチェックポイントがありません"

## Tips

1. **メモリ不足の場合**: バッチサイズを減らしてください
   ```python
   --values data.offline_wall_config.batch_size=64
   ```

2. **高速デバッグ**: `quick_debug=True` を使用
   ```python
   --values quick_debug=True
   ```

3. **WandBを無効化**: インターネット接続が不安定な場合
   ```python
   --values wandb=False
   ```

4. **セッションタイムアウト対策**: 長時間のトレーニングの場合、定期的にチェックポイントを保存し、Google Driveにバックアップしてください。

5. **MiniGrid環境の評価**: 選択されている設定ファイル（[level1_test.yaml](pldm/configs/minigrid/level1_test.yaml)）を使用する場合:
   ```python
   !python pldm/train.py --config pldm/configs/minigrid/level1_test.yaml
   ```

## トラブルシューティング

### CUDA out of memory エラー
- バッチサイズを減らす
- モデルサイズを小さくする
- Runtime → Factory reset runtime を実行

### パッケージのインストールエラー
- Runtimeを再起動: Runtime → Restart runtime
- 再度インストールセルを実行

### データセットのダウンロード失敗
- インターネット接続を確認
- スクリプトを再実行

詳細は [GitHub Issues](https://github.com/vladisai/PLDM/issues) を確認してください。